# Triton을 이용하여 Softmax와 Dropout 연산을 하나의 커널로 퓨전하여 속도 측정

> 본 노트북은 SYLLABUS.md에 기반하여 자동 생성된 뼈대(Skeleton) 파일입니다. 상세한 이론, 수식 및 코드는 추가로 구현되어야 합니다.

## 1. 개요 및 학습 목표
이 노트북에서는 해당 주제에 대한 핵심 개념을 다룹니다.

## 2. 핵심 이론 및 수학적 원리
- 수식 및 상세한 동작 원리를 여기에 기록합니다.

## 3. 실습 코드 구현
아래 셀을 통해 파이썬 및 관련 프레임워크 코드를 직접 작성하고 실행해 볼 수 있습니다.

In [ ]:
# 실습을 위한 기본 라이브러리 임포트
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")


---
## Q1: Softmax 수치 안정성 구현

표준 Softmax는 큰 값이 있을 때 수치 불안정 문제가 발생합니다. 수치 안정적인(numerically stable) Softmax를 NumPy로 구현하고 비교하세요.

**요구사항:**
- `softmax_naive(x)`: 단순 `exp(x) / sum(exp(x))` 구현
- `softmax_stable(x)`: `exp(x - max(x)) / sum(exp(x - max(x)))` 구현
- `x = np.array([1000.0, 1001.0, 1002.0])` 입력으로 두 버전을 비교하세요.
- naive 버전에서 NaN/Inf가 발생하는 이유를 주석으로 설명하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import numpy as np

def softmax_naive(x):
    """수치 불안정: 큰 값에서 exp overflow → inf, nan 발생"""
    return np.exp(x) / np.sum(np.exp(x))

def softmax_stable(x):
    """수치 안정: max를 빼서 최대값을 0으로 만든 후 exp 적용"""
    x_shifted = x - np.max(x)  # 최대값 빼기
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x)

# 테스트
x_large = np.array([1000.0, 1001.0, 1002.0])
x_normal = np.array([1.0, 2.0, 3.0])

print("=== 큰 값 테스트 ===")
print(f"Naive: {softmax_naive(x_large)}")   # NaN 또는 [0, 0, 1] 불안정
print(f"Stable: {softmax_stable(x_large)}")  # 정확한 결과

print("\n=== 일반 값 테스트 ===")
print(f"Naive: {softmax_naive(x_normal)}")
print(f"Stable: {softmax_stable(x_normal)}")
print(f"두 결과 동일 여부: {np.allclose(softmax_naive(x_normal), softmax_stable(x_normal))}")
```
</details>

---
## Q2: Fused Softmax-Dropout 구현

별도로 실행하던 Softmax와 Dropout을 하나의 함수(Fused 연산)로 합쳐 구현하세요. 이는 Triton 커널의 핵심 아이디어입니다.

**요구사항:**
- `fused_softmax_dropout(x, dropout_rate=0.1, training=True)` 함수를 구현하세요.
- 내부에서: 수치 안정 Softmax → Dropout mask 생성 → 마스킹 → 스케일링(`/ (1 - dropout_rate)`) 순서로 처리하세요.
- `training=False`일 때는 Dropout 없이 Softmax만 반환하세요.
- Unfused (두 번의 별도 연산) 방식과 결과를 비교하여 동일성을 검증하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import numpy as np

def fused_softmax_dropout(x, dropout_rate=0.1, training=True, seed=42):
    # Step 1: Softmax (수치 안정)
    x_shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x_shifted)
    softmax_out = exp_x / np.sum(exp_x, axis=-1, keepdims=True)

    if not training:
        return softmax_out

    # Step 2: Dropout (Fused)
    rng = np.random.default_rng(seed)
    mask = rng.random(softmax_out.shape) > dropout_rate
    out = softmax_out * mask / (1.0 - dropout_rate)  # 스케일 보정
    return out

# 테스트 입력
np.random.seed(42)
x = np.random.randn(4, 8).astype(np.float32)  # (batch=4, seq_len=8)

fused_out = fused_softmax_dropout(x, dropout_rate=0.1, training=True)
print(f"Fused 출력 shape: {fused_out.shape}")
print(f"합산=1 확인 (Dropout 없을 때): {np.allclose(fused_softmax_dropout(x, training=False).sum(axis=-1), 1.0)}")
print(f"Fused 출력 예시 (행 0): {fused_out[0].round(4)}")
```
</details>

---
## Q3: 메모리 접근 횟수 비교 분석

Fused 커널의 핵심 이점은 메모리 접근 횟수(Memory Access) 감소입니다. Unfused vs Fused 방식의 이론적 메모리 트래픽을 계산하세요.

**요구사항:**
- `seq_len = 2048`, `num_heads = 32`, `batch = 8`, dtype = float16 (2 bytes) 기준으로 설정하세요.
- **Unfused**: Softmax 결과를 HBM에 저장 후 Dropout이 다시 읽음 = Read + Write + Read 3회
- **Fused**: 한 번 읽어서 처리 후 한 번 씀 = Read + Write 2회
- 각 방식의 총 바이트를 계산하고 절감율(%)을 출력하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
batch = 8
num_heads = 32
seq_len = 2048
dtype_bytes = 2  # float16

tensor_size = batch * num_heads * seq_len * seq_len * dtype_bytes
tensor_gb = tensor_size / (1024**3)

print(f"Attention 행렬 크기: {tensor_gb:.2f} GB")
print(f"  (batch={batch}, heads={num_heads}, seq={seq_len}x{seq_len})\n")

# Unfused: 3번의 HBM 접근
unfused_traffic = 3 * tensor_size
print(f"Unfused (Softmax→저장→Dropout 읽기):")
print(f"  총 메모리 트래픽: {unfused_traffic / (1024**3):.2f} GB")

# Fused: 2번의 HBM 접근
fused_traffic = 2 * tensor_size
print(f"\nFused (한 번에 처리):")
print(f"  총 메모리 트래픽: {fused_traffic / (1024**3):.2f} GB")

reduction = (1 - fused_traffic / unfused_traffic) * 100
print(f"\n메모리 트래픽 절감: {reduction:.1f}%")
```
</details>